In [1]:
import json
from pathlib import Path

# Load both model JSONs
with open('nhanes_data/diabetes_model.json') as f:
    dm = json.load(f)
with open('nhanes_data/cvd_model.json') as f:
    cm = json.load(f)

# Build the combined weights file for the browser
# This is what js/ml-scorer.js will load
ml_weights = {
    'version': '1.0',
    'trained_on': 'NHANES 2011-2018, non-Hispanic Asian subsample validation',
    'outcome_diabetes': 'HbA1c >= 5.7% (prediabetes threshold)',
    'outcome_cvd': 'non-HDL cholesterol > 130 mg/dL',
    'disclaimer': 'Educational tool only. Not a clinical instrument.',
    'feature_cols': dm['feature_cols'],
    'diabetes': {
        'scaler_mean':  dm['scaler_mean'],
        'scaler_scale': dm['scaler_scale'],
        'coefficients': dm['coefficients'],
        'intercept':    dm['intercept'],
        'test_auc':     dm['test_auc'],
        'cv_auc':       dm['cv_auc_mean'],
        'asian_auc':    dm.get('asian_auc'),
    },
    'cvd': {
        'scaler_mean':  cm['scaler_mean'],
        'scaler_scale': cm['scaler_scale'],
        'coefficients': cm['coefficients'],
        'intercept':    cm['intercept'],
        'test_auc':     cm['test_auc'],
        'cv_auc':       cm['cv_auc_mean'],
        'asian_auc':    cm.get('asian_auc'),
    }
}

# Save to the project's data folder (not ml/)
# This is the file the web app will load
out_path = Path('../data/ml_weights.json')
with open(out_path, 'w') as f:
    json.dump(ml_weights, f, indent=2)

print(f'Saved: {out_path.resolve()}')
print(f'File size: {out_path.stat().st_size / 1024:.1f} KB')
print('\nFeature columns in order:')
for i, col in enumerate(ml_weights['feature_cols']):
    print(f'  {i}: {col}')
print('\nDiabetes model AUC:', ml_weights['diabetes']['test_auc'])
print('CVD model AUC:      ', ml_weights['cvd']['test_auc'])
print('\nWeights file ready for browser inference.')

# Verify: run a quick inference in Python to match what the JS will do
import numpy as np

def js_style_inference(weights_block, features_dict, feature_cols):
    x = np.array([features_dict.get(c, 0) for c in feature_cols])
    x_scaled = (x - np.array(weights_block['scaler_mean'])) / np.array(weights_block['scaler_scale'])
    logit = np.dot(x_scaled, weights_block['coefficients']) + weights_block['intercept']
    return round(1 / (1 + np.exp(-logit)) * 100)

test_features = {
    'feature_glycemic_load': 280,
    'feature_refined_carb_share': 0.85,
    'feature_fiber_per_1000kcal': 4.0,
    'feature_protein_pct_energy': 0.10,
    'feature_sfa_pct_energy': 0.14,
    'feature_mufa_sfa_ratio': 0.4,
    'feature_sodium_mg': 3500,
}
feature_cols = ml_weights['feature_cols']
d_score = js_style_inference(ml_weights['diabetes'], test_features, feature_cols)
c_score = js_style_inference(ml_weights['cvd'],      test_features, feature_cols)
print(f'\nVerification (high-risk day):')
print(f'  Diabetes ML score: {d_score}/100')
print(f'  CVD ML score:      {c_score}/100')
print('These should be >= 60 for a high-risk dietary profile.')


Saved: C:\Users\obbha\OneDrive\South-asian-diet-risk.cd\data\ml_weights.json
File size: 2.5 KB

Feature columns in order:
  0: feature_age
  1: feature_bmi
  2: feature_glycemic_load
  3: feature_refined_carb_share
  4: feature_fiber_per_1000kcal
  5: feature_protein_pct_energy
  6: feature_sfa_pct_energy
  7: feature_mufa_sfa_ratio
  8: feature_sodium_mg

Diabetes model AUC: 0.7674320920156494
CVD model AUC:       0.6694378875145564

Weights file ready for browser inference.

Verification (high-risk day):
  Diabetes ML score: 0/100
  CVD ML score:      4/100
These should be >= 60 for a high-risk dietary profile.


In [2]:
import json, pickle, numpy as np, os
from pathlib import Path

# Load the correct final GBM models
with open('nhanes_data/diabetes_final.pkl', 'rb') as f:
    gb_d = pickle.load(f)
with open('nhanes_data/cvd_final.pkl', 'rb') as f:
    gb_c = pickle.load(f)
with open('nhanes_data/diabetes_final_meta.json') as f:
    dm = json.load(f)
with open('nhanes_data/cvd_final_meta.json') as f:
    cm = json.load(f)

FEATURE_COLS = dm['feature_cols']
print('Features:', FEATURE_COLS)
print('Diabetes model n_estimators:', gb_d.n_estimators)
print('CVD model n_estimators:', gb_c.n_estimators)

def export_tree(tree):
    t = tree.tree_
    def recurse(node):
        if t.children_left[node] == -1:
            return {'leaf': float(t.value[node][0][0])}
        return {
            'feature': int(t.feature[node]),
            'threshold': float(t.threshold[node]),
            'left': recurse(t.children_left[node]),
            'right': recurse(t.children_right[node]),
        }
    return recurse(0)

def export_gbm_model(model, meta, label):
    print(f'Exporting {label}: {len(model.estimators_)} stages...')
    trees = []
    for stage in model.estimators_:
        stage_trees = [export_tree(tree) for tree in stage]
        trees.append(stage_trees)
    return {
        'label': label,
        'feature_cols': meta['feature_cols'],
        'n_estimators': int(model.n_estimators),
        'learning_rate': float(model.learning_rate),
        'init_score': float(model.init_.class_prior_[1]),
        'calibration': meta['calibration'],
        'test_auc': meta['test_auc'],
        'asian_auc': meta.get('asian_auc'),
        'ui_description': meta.get('ui_description', ''),
        'trees': trees,
    }

diabetes_export = export_gbm_model(gb_d, dm, 'diabetes')
cvd_export      = export_gbm_model(gb_c, cm, 'cvd')

print(f'Diabetes trees exported: {len(diabetes_export["trees"])} stages')
print(f'CVD trees exported: {len(cvd_export["trees"])} stages')

ml_weights = {
    'version': '2.0',
    'model_type': 'gradient_boosting',
    'purpose': 'personal_risk_context',
    'trained_on': 'NHANES 2011-2018',
    'feature_cols': FEATURE_COLS,
    'diabetes': diabetes_export,
    'cvd': cvd_export,
}

out_path = Path('../data/ml_weights.json')
with open(out_path, 'w') as f:
    json.dump(ml_weights, f)

size_kb = out_path.stat().st_size / 1024
print(f'\nSaved: {out_path.resolve()}')
print(f'File size: {size_kb:.0f} KB')

# Verify round-trip inference
def js_inference(weights_block, features_dict):
    feature_cols = weights_block['feature_cols']
    x = [features_dict.get(c, 0) for c in feature_cols]
    score = weights_block['init_score']
    lr = weights_block['learning_rate']
    for stage in weights_block['trees']:
        for tree in stage:
            node = tree
            while 'leaf' not in node:
                if x[node['feature']] <= node['threshold']:
                    node = node['left']
                else:
                    node = node['right']
            score += lr * node['leaf']
    prob = 1 / (1 + np.exp(-score))
    p5  = weights_block['calibration']['p5']
    p95 = weights_block['calibration']['p95']
    return int(np.clip(round(10 + (prob - p5) / (p95 - p5) * 80), 0, 100))

test_high = {
    'feature_age': 55, 'feature_bmi': 30, 'feature_sedentary_hrs': 10,
    'feature_glycemic_load': 280, 'feature_refined_carb_share': 0.90,
    'feature_fiber_per_1000kcal': 3, 'feature_protein_pct_energy': 0.08,
    'feature_sfa_pct_energy': 0.14, 'feature_mufa_sfa_ratio': 0.4,
    'feature_sodium_mg': 3500,
}
test_low = {
    'feature_age': 30, 'feature_bmi': 21, 'feature_sedentary_hrs': 4,
    'feature_glycemic_load': 90, 'feature_refined_carb_share': 0.05,
    'feature_fiber_per_1000kcal': 18, 'feature_protein_pct_energy': 0.18,
    'feature_sfa_pct_energy': 0.04, 'feature_mufa_sfa_ratio': 2.8,
    'feature_sodium_mg': 1200,
}

d_high = js_inference(ml_weights['diabetes'], test_high)
d_low  = js_inference(ml_weights['diabetes'], test_low)
c_high = js_inference(ml_weights['cvd'], test_high)
c_low  = js_inference(ml_weights['cvd'], test_low)

print(f'\nVerification:')
print(f'High-risk — Diabetes: {d_high}  CVD: {c_high}')
print(f'Low-risk  — Diabetes: {d_low}   CVD: {c_low}')
print(f'Directionality correct: {d_high > d_low and c_high > c_low}')
print(f'File size: {size_kb:.0f} KB (should be several MB for a real GBM export)')

Features: ['feature_age', 'feature_bmi', 'feature_sedentary_hrs', 'feature_glycemic_load', 'feature_refined_carb_share', 'feature_fiber_per_1000kcal', 'feature_protein_pct_energy', 'feature_sfa_pct_energy', 'feature_mufa_sfa_ratio', 'feature_sodium_mg']
Diabetes model n_estimators: 300
CVD model n_estimators: 300
Exporting diabetes: 300 stages...
Exporting cvd: 300 stages...
Diabetes trees exported: 300 stages
CVD trees exported: 300 stages

Saved: C:\Users\obbha\OneDrive\South-asian-diet-risk.cd\data\ml_weights.json
File size: 815 KB

Verification:
High-risk — Diabetes: 68  CVD: 62
Low-risk  — Diabetes: 16   CVD: 19
Directionality correct: True
File size: 815 KB (should be several MB for a real GBM export)
